**Objective** -- This notebook builds an end-to-end sklearn pipeline to classify countries into three development tiers: Developed, Mid Developed, and Low Development, based on socio-economic indicators like child mortality, income, and life expectancy, so that aid and resources can be directed to the countries that need them most.

**Dataset** -- Unsupervised Learning on Country Data (Kaggle)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

In [ ]:
# Load the dataset
df = pd.read_csv("Country-data.csv")
df.columns = df.columns.str.strip()
df.head()

### Shape, Size and Basic Info of Data

In [ ]:
print("Shape:", df.shape, end='\n\n')
print("Null Values:\n", df.isnull().sum(), end='\n\n')
print("Duplicated Rows:", df.duplicated().sum(), end='\n\n')
print("Data Types:\n", df.dtypes)

In [ ]:
numeric_cols = ["child_mort", "exports", "health", "imports", "income",
                "inflation", "life_expec", "total_fer", "gdpp"]
categorical_cols = ["country"]

### Skewness and Kurtosis Analysis

In [ ]:
for col in numeric_cols:
    skew = df[col].skew()
    kurt = df[col].kurt()
    print(f"{col}: skewness = {skew:.3f} | kurtosis = {kurt:.3f}")
    

Almost all features are positively skewed — most countries have low values but a few extreme ones pull the distribution to the right. Inflation is the most extreme case (skewness ~5.15, kurtosis ~41.74). This confirms that outlier handling via **Winsorization** is essential before feeding data into our pipeline.

### Histograms (Distribution of Each Feature)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col], bins=40, edgecolor='white', color='steelblue')
    axes[i].set_title(col, fontsize=13)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Frequency")
plt.suptitle("Feature Distributions", fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

### Boxplots (Outlier Detection)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    axes[i].boxplot(df[col], patch_artist=True,
                    boxprops={'facecolor': 'steelblue', 'alpha': 0.6})
    axes[i].set_title(col, fontsize=13)
plt.suptitle("Boxplots — Outlier View", fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

### Correlation Heatmap

In [ ]:
# Correlation matrix as heatmap (more visual than raw table)
corr = df[numeric_cols].corr()
plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, square=True)
plt.title("Feature Correlation Heatmap", fontsize=14)
plt.tight_layout()
plt.show()

### Custom Winsorizer (Outlier Capping)

In [ ]:
class Winsorizer(BaseEstimator, TransformerMixin):
    """Caps outliers at 1st and 99th percentile."""
    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.lower_ = X.quantile(0.01)
        self.upper_ = X.quantile(0.99)
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        return X.clip(lower=self.lower_, upper=self.upper_, axis=1)

In [ ]:
X = df[numeric_cols]
X.head()

### Elbow Method — Finding Optimal K

In [ ]:
preprocessing_pipeline = Pipeline([
    ('imputer',    SimpleImputer(strategy='median')),
    ('winsorizer', Winsorizer()),
    ('scaler',     StandardScaler())
])

X_scaled = preprocessing_pipeline.fit_transform(X)
inertia_values = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertia_values.append(km.inertia_)
    print(f"k={k}  →  inertia = {km.inertia_:.2f}")

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(list(k_range), inertia_values, marker='o', color='steelblue', linewidth=2)
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Graph — Optimal K Selection")
plt.xticks(list(k_range))
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### Final Pipeline with KMeans (k=3)

In [ ]:
best_k = 3
final_pipeline = Pipeline([
    ('imputer',    SimpleImputer(strategy='median')),
    ('winsorizer', Winsorizer()),
    ('scaler',     StandardScaler()),
    ('kmeans',     KMeans(n_clusters=best_k, random_state=42, n_init=10))
])
final_pipeline.fit(X)

### Cluster Assignment and Silhouette Score

In [ ]:
df['cluster'] = final_pipeline.predict(X)
sil_score = silhouette_score(X_scaled, df['cluster'])
print(f"Silhouette Score (k={best_k}): {sil_score:.4f}")
print("\nCluster Distribution:")
print(df['cluster'].value_counts().sort_index())

### DBSCAN — Initial Run

In [ ]:
dbscan_init = DBSCAN(eps=1.5, min_samples=5)
dbscan_labels = dbscan_init.fit_predict(X_scaled)
df['dbscan_cluster'] = dbscan_labels

n_clusters_init = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise_init = list(dbscan_labels).count(-1)

print(f"DBSCAN clusters found: {n_clusters_init}")
print(f"Noise points (-1): {n_noise_init}")
print("\nDistribution:")
print(pd.Series(dbscan_labels).value_counts().sort_index())

### DBSCAN Grid Search — Hyperparameter Tuning
Only one cluster found above, so we need to optimize eps and min_samples.

In [ ]:
grid_results = []
for eps in [0.3, 0.5, 0.8, 1.0, 1.2, 1.5, 2.0]:
    for min_s in [3, 4, 5, 7, 10]:
        labels = DBSCAN(eps=eps, min_samples=min_s).fit_predict(X_scaled)
        n_cls = len(set(labels)) - (1 if -1 in labels else 0)
        n_nse = list(labels).count(-1)
        score = silhouette_score(X_scaled, labels) if n_cls >= 2 else -999
        grid_results.append({
            'eps': eps, 'min_samples': min_s,
            'n_clusters': n_cls, 'n_noise': n_nse,
            'silhouette': round(score, 4)
        })

res_df = pd.DataFrame(grid_results)
valid_df = res_df[res_df['n_clusters'] >= 2].sort_values('silhouette', ascending=False)
print(valid_df.head(10).to_string(index=False))

In [ ]:
best_params = valid_df.iloc[0]
print(f"Best eps={best_params['eps']}, min_samples={best_params['min_samples']}")
print(f"Clusters: {best_params['n_clusters']}, Noise: {best_params['n_noise']}, "
      f"Silhouette: {best_params['silhouette']}")

best_dbscan = DBSCAN(eps=best_params['eps'], min_samples=int(best_params['min_samples']))
dbscan_labels = best_dbscan.fit_predict(X_scaled)
df['dbscan_cluster'] = dbscan_labels

print(f"\nDBSCAN clusters found: {len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)}")
print(f"Noise points (-1): {list(dbscan_labels).count(-1)}")
print("\nDistribution:")
print(pd.Series(dbscan_labels).value_counts().sort_index())

### Model Comparison — KMeans vs DBSCAN

In [ ]:
kmeans_sil = silhouette_score(X_scaled, df['cluster'])
dbscan_sil = silhouette_score(X_scaled, dbscan_labels)
print(f"KMeans Silhouette Score : {kmeans_sil:.4f}")
print(f"DBSCAN Silhouette Score : {dbscan_sil:.4f}")

# Bar chart comparison
plt.figure(figsize=(6, 4))
plt.bar(['KMeans', 'DBSCAN'], [kmeans_sil, dbscan_sil],
        color=['steelblue', 'coral'], edgecolor='white', width=0.4)
plt.ylabel("Silhouette Score")
plt.title("KMeans vs DBSCAN — Silhouette Score Comparison")
plt.ylim(0, 0.5)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

KMeans achieved a silhouette score of ~0.28 vs DBSCAN at ~0.19 — confirming KMeans is better suited for this dataset.

### PCA Visualization (2D)

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
ev = pca.explained_variance_ratio_
print(f"Variance explained → PC1: {ev[0]:.2%} | PC2: {ev[1]:.2%} | Total: {sum(ev):.2%}")

cluster_colors = {0: 'green', 1: 'red', 2: 'blue'}
plt.figure(figsize=(10, 7))
for cid in sorted(df['cluster'].unique()):
    mask = df['cluster'] == cid
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                c=cluster_colors[cid], label=f"Cluster {cid}",
                alpha=0.75, edgecolors='white', linewidth=0.5, s=80)

plt.xlabel(f"PC1 ({ev[0]:.1%} variance)")
plt.ylabel(f"PC2 ({ev[1]:.1%} variance)")
plt.title("PCA — 2D Cluster Visualization")
plt.legend(title="Cluster")
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

### Cluster Summary Table

In [ ]:
cluster_summary = df.groupby('cluster')[numeric_cols].mean().round(2)
print(cluster_summary.T.to_string())

In [ ]:
cluster_summary.T.style.background_gradient(cmap='RdYlGn', axis=1)

### Predict Function

In [ ]:
CLUSTER_LABELS = {
    0: "Developed",
    1: "Mid Developed",
    2: "Low Development"
}

def predict_cluster(user_input: dict) -> int:
    """
    Predict development tier for a country given socio-economic indicators.
    Pass any subset of numeric_cols; missing values are handled via imputer.
    """
    row = {col: np.nan for col in numeric_cols}
    for key, val in user_input.items():
        if key in numeric_cols:
            row[key] = val

    input_df = pd.DataFrame([row], columns=numeric_cols)
    cluster_id = final_pipeline.predict(input_df)[0]

    print(f"Predicted Cluster  : {cluster_id}")
    print(f"Development Tier   : {CLUSTER_LABELS.get(cluster_id, 'Unknown')}")
    return cluster_id

### Test Cases

In [ ]:
# Test 1: Low development country
predict_cluster({'child_mort': 90, 'exports': 10, 'health': 3, 'imports': 15,
                 'income': 1500, 'inflation': 12, 'life_expec': 55,
                 'total_fer': 5.2, 'gdpp': 1200})

In [ ]:
# Test 2: Developed country (partial input)
predict_cluster({'income': 45000, 'gdpp': 38000, 'life_expec': 79})

In [ ]:
# Test 3: Edge case / extreme values
predict_cluster({'child_mort': 999, 'income': -5000, 'gdpp': 150000})

### Key Observations

**Observation 1** — Cluster 2 is the high-mortality, low-development group. Child mortality reaches ~93, life expectancy is only ~59 years, income ~3942, and fertility rate ~5.01. These countries need the most urgent humanitarian aid and healthcare investment.

**Observation 2** — Cluster 0 represents the top-tier economic zone: GDP per capita ~42494, income ~45672, child mortality near zero (~5), and life expectancy ~80 years — the most prosperous nations.

**Observation 3** — Cluster 1 sits in the middle ground: income ~12305, GDP per capita ~6486, child mortality ~21.93, life expectancy ~72.81. These are transitioning economies with room to grow.

**Observation 4** — Inflation tells an important story. Cluster 2 countries average ~12.02% inflation vs ~2.67% in Cluster 0, confirming that economic instability and underdevelopment go hand in hand.